<a href="https://colab.research.google.com/github/Sakith-N/Statistical-Learning-e22252/blob/assignment-7b/Q3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1.

In [ ]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import beta

theta_grid = np.linspace(0.01, 1.0, 500)
prior_pdf = beta.pdf(theta_grid, 8, 1.5)

fig = go.Figure(go.Scatter(x=theta_grid, y=prior_pdf, name="Prior PDF"))
fig.update_layout(title="Initial Structural Health Prior Beta(8, 1.5)", xaxis_title="Stiffness Factor (theta)", yaxis_title="Density")
fig.show()

2. $\epsilon_k = \ln\left(\frac{y_k}{\theta \cdot K_{\text{nominal}}}\right) = \ln(y_k) - \ln(\theta \cdot K_{\text{nominal}})$


Using the transformation of variables or the definition of a log normal distribution, the conditional PDF for a single continuous inspection measurement $y_k$ is:$L(y_k \mid \theta) = \frac{1}{y_k \sigma \sqrt{2\pi}} \exp\left( -\frac{\left(\ln(y_k) - \ln(\theta \cdot K_{\text{nominal}})\right)^2}{2\sigma^2} \right)$



Assuming independent measurement errors over time, the joint history likelihood function across $k$ inspection steps is the product of individual steps:$L(y^{(k)} \mid \theta) = \prod_{i=1}^k \frac{1}{y_i \sigma \sqrt{2\pi}} \exp\left( -\frac{\left(\ln(y_i) - \ln(\theta \cdot K_{\text{nominal}})\right)^2}{2\sigma^2} \right)$

3. The recursive relationship up to a proportionality constant is:$f_{\Theta \mid Y^{(k)}}(\theta \mid y^{(k)}) \propto \left[ \frac{1}{y_k \sigma \sqrt{2\pi}} \exp\left( -\frac{\left(\ln(y_k) - \ln(\theta \cdot K_{\text{nominal}})\right)^2}{2\sigma^2} \right) \right] \cdot f_{\Theta \mid Y^{(k-1)}}(\theta \mid y^{(k-1)})$

4.


*   Running Posterior Mean:

  $\hat{\theta}_{\text{Bayes}}^{(k)} = \int_{0}^{1} \theta \cdot f_{\Theta \mid Y^{(k)}}(\theta \mid y^{(k)}) \, d\theta$

*   Running Maximum A Posteriori (MAP) Estimate:

  $\hat{\theta}_{\text{MAP}}^{(k)} = \arg\max_{\theta \in (0, 1]} f_{\Theta \mid Y^{(k)}}(\theta \mid y^{(k)})$



5.


In [1]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

np.random.seed(30)
theta_true = 0.68
K_nominal = 50.0
sigma = 0.15
n_sensor_readings = 15

theta_grid = np.linspace(0.01, 1.0, 1000)
current_posterior = stats.beta.pdf(theta_grid, 8, 1.5)
current_posterior /= np.trapezoid(current_posterior, theta_grid)

milestones = [0, 1, 2, 5, 10, 15]
running_bayes = [np.trapezoid(theta_grid * current_posterior, theta_grid)]
running_map = [theta_grid[np.argmax(current_posterior)]]
steps = [0]

fig_curves = go.Figure()
fig_curves.add_trace(go.Scatter(x=theta_grid, y=current_posterior, mode='lines', name='Step 0 (Prior)', line=dict(dash='dash', color='gray')))

for k in range(1, n_sensor_readings + 1):
    noise = np.random.normal(0, sigma)
    y_k = (theta_true * K_nominal) * np.exp(noise)

    likelihood = stats.lognorm.pdf(y_k, s=sigma, scale=theta_grid * K_nominal)
    current_posterior *= likelihood
    current_posterior /= np.trapezoid(current_posterior, theta_grid)

    t_bayes = np.trapezoid(theta_grid * current_posterior, theta_grid)
    t_map = theta_grid[np.argmax(current_posterior)]

    running_bayes.append(t_bayes)
    running_map.append(t_map)
    steps.append(k)

    if k in milestones:
        fig_curves.add_trace(go.Scatter(x=theta_grid, y=current_posterior, mode='lines', name=f'Step {k}'))

fig_curves.add_vline(x=theta_true, line_dash="dot", line_color="red", line_width=2)
fig_curves.update_layout(title="Posterior Density Curve Progression", xaxis_title="Efficiency Factor (θ)", yaxis_title="Density", template="plotly_white")
fig_curves.show()

fig_track = go.Figure()
fig_track.add_hline(y=theta_true, line_dash="dash", line_color="red", line_width=2)
fig_track.add_trace(go.Scatter(x=steps, y=running_bayes, mode='lines+markers', name='Posterior Mean'))
fig_track.add_trace(go.Scatter(x=steps, y=running_map, mode='lines+markers', name='MAP Estimate'))
fig_track.update_layout(title="Structural Efficiency Estimator Tracking", xaxis_title="Sensor Reading Milestone (k)", yaxis_title="Estimated Efficiency", template="plotly_white")
fig_track.show()